In [1]:
import os
import pandas as pd
from math import sqrt
import numpy as np
from sklearn.linear_model import Ridge
from sklearn import linear_model

from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse

## Interaction-based: Focus on interactions between users and items
Aka Collaborative filtering 

In [2]:
## Utilize schema from readme
u_data_cols = ['user_id', 'item_id', 'rating', 'timestamp']
u_item_cols = ['item_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL', 'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
u_user_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']

In [3]:
df_users = pd.read_csv('data/u.user', sep='|', names=u_user_cols, encoding='latin-1')
df_items = pd.read_csv('data/u.item', sep='|', names=u_item_cols, encoding='latin-1')
df_ratings_base = pd.read_csv('data/ua.base', sep='\t', names=u_data_cols, encoding='latin-1')
df_ratings_test = pd.read_csv('data/ua.test', sep='\t', names=u_data_cols, encoding='latin-1')

In [4]:
class CF(object):
    """docstring for CF"""
    def __init__(self, Y_data, k, dist_func = cosine_similarity, uuCF = 1):
        self.uuCF = uuCF # user-user (1) or item-item (0) CF
        self.Y_data = Y_data if uuCF else Y_data[:, [1, 0, 2]]
        self.k = k
        self.dist_func = dist_func
        self.Ybar_data = None
        # number of users and items. Remember to add 1 since id starts from 0
        self.n_users = int(np.max(self.Y_data[:, 0])) + 1 
        self.n_items = int(np.max(self.Y_data[:, 1])) + 1
    
    def add(self, new_data):
        """
        Update Y_data matrix when new ratings come.
        For simplicity, suppose that there is no new user or item.
        """
        self.Y_data = np.concatenate((self.Y_data, new_data), axis = 0)
    
    def normalize_Y(self):
        users = self.Y_data[:, 0] # all users - first col of the Y_data
        self.Ybar_data = self.Y_data.copy()
        self.mu = np.zeros((self.n_users,))
        for n in range(self.n_users):
            # row indices of rating done by user n
            # since indices need to be integers, we need to convert
            ids = np.where(users == n)[0].astype(np.int32)
            # indices of all ratings associated with user n
            item_ids = self.Y_data[ids, 1] 
            # and the corresponding ratings 
            ratings = self.Y_data[ids, 2]
            # take mean
            m = np.mean(ratings) 
            if np.isnan(m):
                m = 0 # to avoid empty array and nan value
            self.mu[n] = m
            # normalize
            self.Ybar_data[ids, 2] = ratings - self.mu[n]

        ################################################
        # form the rating matrix as a sparse matrix. Sparsity is important 
        # for both memory and computing efficiency. For example, if #user = 1M, 
        # #item = 100k, then shape of the rating matrix would be (100k, 1M), 
        # you may not have enough memory to store this. Then, instead, we store 
        # nonzeros only, and, of course, their locations.
        self.Ybar = sparse.coo_matrix((self.Ybar_data[:, 2],
            (self.Ybar_data[:, 1], self.Ybar_data[:, 0])), (self.n_items, self.n_users))
        self.Ybar = self.Ybar.tocsr()

    def similarity(self):
        eps = 1e-6
        self.S = self.dist_func(self.Ybar.T, self.Ybar.T)
    
        
    def refresh(self):
        """
        Normalize data and calculate similarity matrix again (after
        some few ratings added)
        """
        self.normalize_Y()
        self.similarity() 
        
    def fit(self):
        self.refresh()
        
    
    def __pred(self, u, i, normalized = 1):
        """ 
        predict the rating of user u for item i (normalized)
        if you need the un
        """
        # Step 1: find all users who rated i
        ids = np.where(self.Y_data[:, 1] == i)[0].astype(np.int32)
        # Step 2: 
        users_rated_i = (self.Y_data[ids, 0]).astype(np.int32)
        # Step 3: find similarity btw the current user and others 
        # who already rated i
        sim = self.S[u, users_rated_i]
        # Step 4: find the k most similarity users
        a = np.argsort(sim)[-self.k:] 
        # and the corresponding similarity levels
        nearest_s = sim[a]
        # How did each of 'near' users rated item i
        r = self.Ybar[i, users_rated_i[a]].toarray().ravel()
        if normalized:
            # add a small number, for instance, 1e-8, to avoid dividing by 0
            return np.dot(r, nearest_s)/(np.abs(nearest_s).sum() + 1e-8)

        return np.dot(r, nearest_s)/(np.abs(nearest_s).sum() + 1e-8) + self.mu[u]
    
    def pred(self, u, i, normalized = 1):
        """ 
        predict the rating of user u for item i (normalized)
        if you need the un
        """
        if self.uuCF: return self.__pred(u, i, normalized)
        return self.__pred(i, u, normalized)
            
    
    def recommend(self, u):
        """
        Determine all items should be recommended for user u.
        The decision is made based on all i such that:
        self.pred(u, i) > 0. Suppose we are considering items which 
        have not been rated by u yet. 
        """
        ids = np.where(self.Y_data[:, 0] == u)[0]
        items_rated_by_u = self.Y_data[ids, 1].tolist()              
        recommended_items = []
        for i in range(self.n_items):
            if i not in items_rated_by_u:
                rating = self.__pred(u, i)
                if rating > 0: 
                    recommended_items.append(i)
        
        return recommended_items 
    
    def recommend2(self, u):
        """
        Determine all items should be recommended for user u.
        The decision is made based on all i such that:
        self.pred(u, i) > 0. Suppose we are considering items which 
        have not been rated by u yet. 
        """
        ids = np.where(self.Y_data[:, 0] == u)[0]
        items_rated_by_u = self.Y_data[ids, 1].tolist()              
        recommended_items = []
    
        for i in range(self.n_items):
            if i not in items_rated_by_u:
                rating = self.__pred(u, i)
                if rating > 0: 
                    recommended_items.append(i)
        
        return recommended_items 

    def print_recommendation(self):
        """
        print all items which should be recommended for each user 
        """
        print('Recommendation: ')
        for u in range(self.n_users):
            recommended_items = self.recommend(u)
            if self.uuCF:
                print('    Recommend item(s):', recommended_items, 'for user', u)
            else: 
                print('    Recommend item', u, 'for user(s) : ', recommended_items)

In [6]:
rate_train = df_ratings_base.values
rate_test = df_ratings_test.values

# indices start from 0
rate_train[:, :2] -= 1
rate_test[:, :2] -= 1

In [8]:
## User-User CF

rs = CF(rate_train, k = 30, uuCF = 1)
rs.fit()

n_tests = rate_test.shape[0]
SE = 0 # squared error
for n in range(n_tests):
    pred = rs.pred(int(rate_test[n, 0]), int(rate_test[n, 1]), normalized = 0)
    SE += (pred - rate_test[n, 2])**2 

RMSE = np.sqrt(SE/n_tests)
print('User-user CF, RMSE =', RMSE)

User-user CF, RMSE = 0.9767347622900232


In [12]:
rate_train

array([[        0,         0,         5, 874965758],
       [        0,         1,         3, 876893171],
       [        0,         2,         4, 878542960],
       ...,
       [      942,      1187,         3, 888640250],
       [      942,      1227,         3, 888640275],
       [      942,      1329,         3, 888692465]], dtype=int64)

In [ ]:
rs = CF(rate_train, k = 30, uuCF = 1)


In [9]:
## Item-Item CF

rs = CF(rate_train, k = 30, uuCF = 0)
rs.fit()

n_tests = rate_test.shape[0]
SE = 0 # squared error
for n in range(n_tests):
    pred = rs.pred(int(rate_test[n, 0]), int(rate_test[n, 1]), normalized = 0)
    SE += (pred - rate_test[n, 2])**2 

RMSE = np.sqrt(SE/n_tests)
print('Item-item CF, RMSE =', RMSE)

c:\Users\vncpyy7h\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\vncpyy7h\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Item-item CF, RMSE = 0.9678464362651048


In [13]:
import numpy as np

# 1. Create a mock dataset: [user_id, item_id, rating]
# User 0 has NOT rated Item 2
# User 2 has NOT rated Item 1
rate_train = np.array([
    [0, 0, 5.0], [0, 1, 3.0], [0, 3, 1.0],
    [1, 0, 4.0], [1, 1, 4.0], [1, 2, 2.0],
    [2, 0, 2.0], [2, 2, 4.0], [2, 4, 5.0],
    [3, 1, 1.0], [3, 3, 4.0], [3, 4, 4.0]
])

# 2. Initialize User-User CF with a neighborhood size of k=2
print("--- Step 1: Initializing the model ---")
rs = CF(rate_train, k=2, uuCF=1)

print(f"Mode: User-User CF (uuCF={rs.uuCF})")
print(f"Neighborhood size (k): {rs.k}")
print(f"Number of users: {rs.n_users}")
print(f"Number of items: {rs.n_items}\n")

# 3. Fit the model (this normalizes ratings and calculates user similarities)
print("--- Step 2: Fitting the model ---")
rs.fit()

print("User mean ratings (mu):")
for u, mean in enumerate(rs.mu):
    print(f"  User {u} average rating: {mean:.2f}")

print("\nUser-to-User Similarity Matrix (self.S):")
print(np.round(rs.S, 2))
print()

# 4. Predict a missing rating
# Let's predict what rating User 0 would give to Item 2 (which they haven't rated)
print("--- Step 3: Making a prediction ---")
predicted_rating = rs.pred(u=0, i=2, normalized=0)
print(f"Predicted raw rating of User 0 for Item 2: {predicted_rating:.2f}\n")

# 5. Generate recommendations for User 0
print("--- Step 4: Generating recommendations ---")
recommendations = rs.recommend(u=0)
print(f"Recommended item IDs for User 0: {recommendations}")

--- Step 1: Initializing the model ---
Mode: User-User CF (uuCF=1)
Neighborhood size (k): 2
Number of users: 4
Number of items: 5

--- Step 2: Fitting the model ---
User mean ratings (mu):
  User 0 average rating: 3.00
  User 1 average rating: 3.33
  User 2 average rating: 3.67
  User 3 average rating: 3.00

User-to-User Similarity Matrix (self.S):
[[ 1.    0.29 -0.55 -0.29]
 [ 0.29  1.   -0.44 -0.33]
 [-0.55 -0.44  1.    0.25]
 [-0.29 -0.33  0.25  1.  ]]

--- Step 3: Making a prediction ---
Predicted raw rating of User 0 for Item 2: 2.32

--- Step 4: Generating recommendations ---
Recommended item IDs for User 0: []


In [15]:
rs.Ybar_data

array([[ 0.        ,  0.        ,  2.        ],
       [ 0.        ,  1.        ,  0.        ],
       [ 0.        ,  3.        , -2.        ],
       [ 1.        ,  0.        ,  0.66666667],
       [ 1.        ,  1.        ,  0.66666667],
       [ 1.        ,  2.        , -1.33333333],
       [ 2.        ,  0.        , -1.66666667],
       [ 2.        ,  2.        ,  0.33333333],
       [ 2.        ,  4.        ,  1.33333333],
       [ 3.        ,  1.        , -2.        ],
       [ 3.        ,  3.        ,  1.        ],
       [ 3.        ,  4.        ,  1.        ]])